In [1]:
import os
import numpy as np
from obspy import read
from tqdm import tqdm

BASE_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready'
OUTPUT_DATASET = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_no_le'

# Buat folder output
os.makedirs(os.path.join(OUTPUT_DATASET, 'gempa'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DATASET, 'noise'), exist_ok=True)

def process_files():
    # Iterasi semua tahun
    for year in os.listdir(BASE_DIR):
        year_path = os.path.join(BASE_DIR, year)
        if not os.path.isdir(year_path): continue
        
        # Iterasi event ID
        for event_id in tqdm(os.listdir(year_path), desc=f"Processing {year}"):
            event_path = os.path.join(year_path, event_id)
            if not os.path.isdir(event_path): continue
            
            for file in os.listdir(event_path):
                if file.endswith('.mseed'):
                    try:
                        st = read(os.path.join(event_path, file))
                        tr = st[0] # Ambil komponen Z
                        
                        # Detrend dan Bandpass
                        tr.detrend('demean').filter('bandpass', freqmin=1.0, freqmax=20.0)
                        
                        # 1. Ambil segmen GEMPA (asumsi P-wave di tengah)
                        mid = len(tr.data) // 2
                        gempa_seg = tr.data[mid-500:mid+500] # 10 detik
                        np.save(os.path.join(OUTPUT_DATASET, 'gempa', f"{event_id}.npy"), gempa_seg)
                        
                        # 2. Ambil segmen NOISE (asumsi awal file sebelum gempa)
                        noise_seg = tr.data[0:1000]
                        np.save(os.path.join(OUTPUT_DATASET, 'noise', f"{event_id}.npy"), noise_seg)
                        
                    except Exception as e:
                        continue

process_files()
print("Dataset processing selesai!")

Processing 2013:   0%|          | 0/351 [00:00<?, ?it/s]/opt/homebrew/Caskroom/miniforge/base/envs/waveform/lib/python3.10/site-packages/obspy/signal/filter.py:87: UserWarning: Selected high corner frequency (20.0) of bandpass is at or above Nyquist (10.0). Applying a high-pass instead.
  warnings.warn(msg)
Processing 2014:  51%|█████     | 321/627 [00:03<00:03, 92.93it/s]/opt/homebrew/Caskroom/miniforge/base/envs/waveform/lib/python3.10/site-packages/obspy/signal/filter.py:87: UserWarning: Selected high corner frequency (20.0) of bandpass is at or above Nyquist (20.0). Applying a high-pass instead.
  warnings.warn(msg)
Processing 2008:  41%|████      | 282/684 [00:03<00:04, 89.81it/s]/opt/homebrew/Caskroom/miniforge/base/envs/waveform/lib/python3.10/site-packages/obspy/signal/filter.py:87: UserWarning: Selected high corner frequency (20.0) of bandpass is at or above Nyquist (9.999970436096191). Applying a high-pass instead.
  warnings.warn(msg)
Processing 2006:   0%|          | 0/451 

Dataset processing selesai!
